# Amazon Recommender System - Data Preprocessing
---

Mục tiêu: chuyển file rating thô thành tập sạch, không trùng lặp, chuẩn hoá thời gian và ánh xạ User/Product sang chỉ số để huấn luyện mô hình khuyến nghị.

Các bước:
1. Đọc dữ liệu thô và kiểm tra dạng.
2. Lọc dòng không hợp lệ (UserId/ProductId rỗng, Rating ngoài [1,5], Timestamp <=0).
3. Loại trùng (giữ rating mới nhất cho mỗi (UserId,ProductId)).
4. **K-core filtering (k=5)**: quay lại tiêu chí chặt hơn để dữ liệu đặc tương đương repo beauty và giảm nhiễu.
5. Chuẩn hoá timestamp: thêm cột datetime (`TimestampDT`) và epoch (`Timestamp`).
6. Tạo mapping liên tục cho User và Product.
7. Feature engineering: chỉ giữ `TimeWeight` (min–max scale timestamp) để dùng cho SGD time-decay, bỏ toàn bộ trọng số/phần dư khác.


In [1]:
import sys
import numpy as np

sys.path.insert(0, '../src')
from data_processing import (
    load_raw_data,
    clean_raw,
    deduplicate_latest,
    to_datetime_minimal,
    encode_ids,
    split_train_test,
    save_processed
)

In [2]:
# 1. Load raw ratings
raw, headers = load_raw_data('../data/raw/ratings_Beauty.csv')
print(f"Loaded raw rows: {len(raw):,}")
print('Columns:', headers)



Loaded raw rows: 2,023,070
Columns: ('UserId', 'ProductId', 'Rating', 'Timestamp')


## 2. Lọc giá trị không hợp lệ

Loại bỏ các dòng có:
- UserId / ProductId rỗng.
- Rating không parse được hoặc ngoài [1,5].
- Timestamp không phải số nguyên dương.

Kết quả: structured array sạch + thống kê nhanh (users, items, density).

In [3]:
# Clean raw data via local function
cleaned = clean_raw(raw)
users = len(np.unique(cleaned['UserId']))
products = len(np.unique(cleaned['ProductId']))
rows = len(cleaned)
density = rows / (users * products)
print(f'Cleaned rows: {rows} | Users: {users} | Products: {products} | Density: {density:.6f}')

data = cleaned

Cleaned rows: 2023070 | Users: 1210271 | Products: 249274 | Density: 0.000007


## 3. Loại trùng lặp

Mỗi (UserId,ProductId) có thể xuất hiện nhiều lần. Giữ bản ghi mới nhất theo Timestamp bằng cách sắp xếp key rồi chọn lần xuất hiện đầu. Giảm nhiễu, tránh double-count rating.

In [4]:
before = len(data)
dedup = deduplicate_latest(data)
removed = before - len(dedup)
print(f'Deduplicated: {before} -> {len(dedup)} (removed {removed})')

data = dedup

Deduplicated: 2023070 -> 2023070 (removed 0)


## 3.5. K-core Filtering (Loại bỏ Noise)
 
Áp dụng k-core filtering để loại bỏ users/items có quá ít interactions:
- **k=5 (mặc định)**:đảm bảo mỗi user/item có ≥5 ratings → dữ liệu đặc, ít nhiễu.
- Có thể giảm xuống 3 nếu cần thêm coverage, nhưng mặc định ưu tiên tín hiệu mạnh để SVD học tốt.
- **Iterative**: Lặp cho đến khi không còn user/item nào bị loại.
- **Mục đích**: 
  - Giảm noise từ users/items hiếm nhưng vẫn kiểm soát coverage.
  - Đảm bảo mỗi điểm dữ liệu đều có lịch sử tối thiểu trước khi build embedding.


In [5]:
def filter_k_core(data, k=5):
    filtered_data = data.copy()
    iteration = 0

    while True:
        iteration += 1
        original_shape = filtered_data.shape[0]

        user_ids, user_counts = np.unique(filtered_data['UserId'], return_counts=True)
        valid_users = user_ids[user_counts >= k]
        mask_users = np.isin(filtered_data['UserId'], valid_users)
        filtered_data = filtered_data[mask_users]

        product_ids, product_counts = np.unique(filtered_data['ProductId'], return_counts=True)
        valid_products = product_ids[product_counts >= k]
        mask_products = np.isin(filtered_data['ProductId'], valid_products)
        filtered_data = filtered_data[mask_products]

        current_shape = filtered_data.shape[0]
        if original_shape == current_shape:
            break
    return filtered_data

print(f"Before k-core: {len(dedup):,} ratings")
data_kcore = filter_k_core(dedup, k=5)
print(f"After  k-core: {len(data_kcore):,} ratings (removed {len(dedup)-len(data_kcore):,})")
# Proceed with datetime conversion on filtered data
minimal = to_datetime_minimal(data_kcore)
data = minimal

Before k-core: 2,023,070 ratings
After  k-core: 198,502 ratings (removed 1,824,568)


## 5. Mapping ID sang chỉ số

Dùng `np.unique(..., return_inverse=True)` để tạo index 0..N-1 cho user và product:
- Giảm bộ nhớ khi xây ma trận/sparse.
- Tương thích với thuật toán (MF, KNN, GNN).

Kết quả: `ratings_mapped` chứa cả ID gốc, index, rating, datetime và epoch.

In [6]:
# Encode IDs -> indices (local)
mapped, user_id_list, product_id_list = encode_ids(data)
print(f'Mapped rows: {len(mapped)} | Users: {len(user_id_list)} | Products: {len(product_id_list)}')

ratings_mapped = mapped

Mapped rows: 198502 | Users: 22363 | Products: 12101


In [7]:
files = save_processed(
    ratings_mapped,
    user_id_list,
    product_id_list,
    '../data/processed',
    use_datetime=True
)
print('Exported files:', {k: v for k, v in files.items() if k != 'stats'})
stats = files.get('stats', {}) or {}
if stats:
    print(
        "Rating mean/std: "
        f"{stats.get('rating_mean', float('nan')):.4f} / {stats.get('rating_std', float('nan')):.4f}"
    )
print(f'Total processed ratings: {len(ratings_mapped):,}')

Exported files: {'ratings': '../data/processed/ratings_processed.csv', 'user_mapping': '../data/processed/user_mapping.csv', 'product_mapping': '../data/processed/product_mapping.csv'}
Rating mean/std: 4.1904 / 1.1666
Total processed ratings: 198,502


## 6. Feature engineering


In [8]:
# Preview processed dataset (showing enriched columns)
preview_lines = 5
with open('../data/processed/ratings_processed.csv', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= preview_lines:
            break



UserId,ProductId,Rating,Timestamp,TimestampDT,TimeWeight,ProductMean
A00414041RD0BXM6WK0GX,B007IY97U0,3.000000,1405296000,2014-07-14T00:00:00,0.998373,4.368421
A00414041RD0BXM6WK0GX,B00870XLDS,2.000000,1405296000,2014-07-14T00:00:00,0.998373,3.333333
A00414041RD0BXM6WK0GX,B008MIRO88,1.000000,1405296000,2014-07-14T00:00:00,0.998373,3.100000
A00414041RD0BXM6WK0GX,B00BQYYMN0,3.000000,1405296000,2014-07-14T00:00:00,0.998373,3.647059
A00414041RD0BXM6WK0GX,B00GRTQBTM,5.000000,1405296000,2014-07-14T00:00:00,0.998373,2.800000
